# Разведочный анализ обучающей выборки (EDA)

Смотрим структуру `train.csv`, размеры изображений и распределение доли площади
правки в масках. Главная цель — понять, как строить валидацию и какие негативы
использовать для метрики AIC.

In [ ]:
import csv
import os
import random
import sys
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRAIN_CSV = ROOT / "train_stage1" / "stage1" / "train.csv"
TRAIN_DIR = ROOT / "train_stage1"
print("корень проекта:", ROOT)

In [ ]:
rows = []
with open(TRAIN_CSV, newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append({
            "chng": r["chng_img_path"].strip(),
            "gt": r["gt_path"].strip(),
            "src": (r.get("orgl_img_path") or "").strip(),
        })

n = len(rows)
distinct_gt = len({r["gt"] for r in rows})
distinct_src = len({r["src"] for r in rows if r["src"]})
empty_src = sum(1 for r in rows if not r["src"])

print(f"строк: {n}")
print(f"уникальных масок: {distinct_gt} (в среднем {n / distinct_gt:.2f} строк на маску)")
print(f"уникальных оригиналов: {distinct_src}")
print(f"строк без оригинала: {empty_src} ({empty_src / n:.1%})")

In [ ]:
rng = random.Random(42)
sample = rng.sample([r["chng"] for r in rows], 1500)
sizes = []
for rel in sample:
    try:
        with Image.open(TRAIN_DIR / rel) as im:
            sizes.append(im.size)
    except OSError:
        pass

widths = np.asarray([s[0] for s in sizes])
heights = np.asarray([s[1] for s in sizes])
print(f"высота: min={heights.min()} p50={int(np.median(heights))} max={heights.max()}")
print(f"ширина: min={widths.min()} p50={int(np.median(widths))} max={widths.max()}")

In [ ]:
sample_masks = rng.sample([r["gt"] for r in rows], 2500)
areas = []
zero_masks = 0
for rel in sample_masks:
    try:
        with Image.open(TRAIN_DIR / rel) as im:
            m = np.asarray(im.convert("L"))
    except OSError:
        continue
    area = float((m > 128).mean())
    areas.append(area)
    zero_masks += area == 0.0

areas = np.asarray(areas)
print(f"доля площади правки: p25={np.percentile(areas,25):.3f} "
      f"p50={np.percentile(areas,50):.3f} p75={np.percentile(areas,75):.3f} max={areas.max():.3f}")
print(f"пустых масок: {zero_masks} из {len(areas)} ({zero_masks/len(areas):.1%})")
print(f"мелких правок (<1% площади): {(areas < 0.01).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(np.log10(areas + 1e-6), bins=40, color="#4c72b0", alpha=0.85)
axes[0].set_xlabel("log10(доля площади правки)")
axes[0].set_ylabel("число масок")
axes[0].set_title("Распределение доли изменённой области")

axes[1].scatter(widths, heights, s=6, alpha=0.4, color="#dd8452")
axes[1].set_xlabel("ширина, px")
axes[1].set_ylabel("высота, px")
axes[1].set_title("Размеры изменённых изображений")

fig.tight_layout()
plt.show()

## Выводы

- **103 699 строк**, из них у 53.5% нет оригинала (только изменённое изображение и маска).
- **Маски переиспользуются**: 87 799 уникальных масок на 103 699 строк → валидацию нужно
  разбивать по группам масок, иначе утечка.
- **~2.4% масок полностью пустые** — это готовые "чистые" негативы для FPR.
- **~5.8% правок занимают меньше 1% кадра** — есть мелкие правки, которые важно не потерять.
- **Размеры разнятся** (примерно от 154×205 до 1024×1024), поэтому изображения ресайзятся
  к фиксированному входу модели, а маска возвращается к исходному разрешению.